In [ ]:
pip install transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 4.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.9.0.13 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu12 9.3.0.75 which is incompatible.
torch 2.6.0+cu124 requires nvid

In [ ]:
import json
import numpy as np
from datasets import Dataset
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments
import torch
from torch.utils.data import DataLoader
from imblearn.over_sampling import RandomOverSampler

def load_data(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    label_map = {"non-sarcasm": 0, "sarcasm": 1}
    texts = [item["caption"] for item in data]
    labels = [label_map[item["label"].strip().lower()] for item in data]
    return texts, labels

def preprocess_dataset(texts, labels, tokenizer):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
    dataset = Dataset.from_dict({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": labels
    })
    return dataset

def oversample_data(texts, labels):
    texts_np = np.array(texts).reshape(-1, 1)
    ros = RandomOverSampler(sampling_strategy="auto")
    texts_resampled, labels_resampled = ros.fit_resample(texts_np, labels)
    return texts_resampled.ravel().tolist(), labels_resampled


def train_with_early_stopping(model, train_dataset, val_dataset, training_args, patience=2):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=training_args.learning_rate)

    train_loader = DataLoader(train_dataset, batch_size=training_args.per_device_train_batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=training_args.per_device_eval_batch_size)

    best_f1 = 0.0
    patience_counter = 0

    for epoch in range(int(training_args.num_train_epochs)):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            inputs = {
                "input_ids": torch.tensor(batch["input_ids"]).to(device),
                "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
                "labels": torch.tensor(batch["labels"]).to(device),
            }

            optimizer.zero_grad()
            outputs = model(**inputs)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"\nEpoch {epoch+1}: Train loss = {total_loss / len(train_loader):.4f}")

        # Đánh giá 
        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in val_loader:
                inputs = {
                    "input_ids": torch.tensor(batch["input_ids"]).to(device),
                    "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
                }
                labels = torch.tensor(batch["labels"]).to(device)

                outputs = model(**inputs)
                preds = torch.argmax(outputs.logits, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        prec = precision_score(all_labels, all_preds, average="macro")
        rec = recall_score(all_labels, all_preds, average="macro")
        f1 = f1_score(all_labels, all_preds, average="macro")

        print(f"Val Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")

        # Early stopping
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

def evaluate_on_test(model, test_dataset, batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            inputs = {
                "input_ids": torch.tensor(batch["input_ids"]).to(device),
                "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
            }
            labels = torch.tensor(batch["labels"]).to(device)
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n===== Test Set Evaluation =====")
    print(classification_report(all_labels, all_preds, target_names=["Non-sarcasm", "Sarcasm"], digits=4))

def main():
    llm_train_path = "/kaggle/input/silver-text/silver_text.json"     
    human_train_path = "/kaggle/input/text-sarcasm/text_train.json"
    val_path = "/kaggle/input/text-sarcasm/text_dev.json"
    test_path = "/kaggle/input/text-sarcasm/text_test.json"

    llm_texts, llm_labels = load_data(llm_train_path)
    human_texts, human_labels = load_data(human_train_path)

    #Oversampling
    human_texts_os, human_labels_os = oversample_data(human_texts, human_labels)
    train_texts = llm_texts + human_texts_os
    train_labels = llm_labels + human_labels_os

    #Load tokenizer và model
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

    train_dataset = preprocess_dataset(train_texts, train_labels, tokenizer)
    val_texts, val_labels = load_data(val_path)
    val_dataset = preprocess_dataset(val_texts, val_labels, tokenizer)

    test_texts, test_labels = load_data(test_path)
    test_dataset = preprocess_dataset(test_texts, test_labels, tokenizer)

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    training_args = TrainingArguments(
        output_dir="./results",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=6,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=10,
        report_to="none"
    )

    #Train
    train_with_early_stopping(model, train_dataset, val_dataset, training_args, patience=2)
    model.load_state_dict(torch.load("best_model.pt"))
    evaluate_on_test(model, test_dataset)

if __name__ == "__main__":
    main()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_84/3486615236.py:55: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(batch["input_ids"]).to(device),
/tmp/ipykernel_84/3486615236.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
/tmp/ipykernel_84/3486615236.py:57: UserWarning: To copy construct from a tensor, it is recommended to use source


Epoch 1: Train loss = 0.6189


/tmp/ipykernel_84/3486615236.py:76: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(batch["input_ids"]).to(device),
/tmp/ipykernel_84/3486615236.py:77: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
/tmp/ipykernel_84/3486615236.py:79: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(batch["labels"]).to(device)


Val Accuracy: 0.7400 | Precision: 0.5200 | Recall: 0.5789 | F1: 0.4902


/tmp/ipykernel_84/3486615236.py:55: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(batch["input_ids"]).to(device),
/tmp/ipykernel_84/3486615236.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
/tmp/ipykernel_84/3486615236.py:57: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(batch["labels"]).to(device),


...


Epoch 3: Train loss = 0.4035


/tmp/ipykernel_84/3486615236.py:76: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(batch["input_ids"]).to(device),
/tmp/ipykernel_84/3486615236.py:77: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
/tmp/ipykernel_84/3486615236.py:79: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(batch["labels"]).to(device)


Val Accuracy: 0.5950 | Precision: 0.5191 | Recall: 0.5974 | F1: 0.4326
Early stopping triggered at epoch 3


/tmp/ipykernel_84/3486615236.py:116: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(batch["input_ids"]).to(device),
/tmp/ipykernel_84/3486615236.py:117: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(batch["attention_mask"]).to(device),
/tmp/ipykernel_84/3486615236.py:119: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(batch["labels"]).to(device)



===== Test Set Evaluation =====
              precision    recall  f1-score   support

 Non-sarcasm     0.9677    0.7895    0.8696       190
     Sarcasm     0.1111    0.5000    0.1818        10

    accuracy                         0.7750       200
   macro avg     0.5394    0.6447    0.5257       200
weighted avg     0.9249    0.7750    0.8352       200

